# CIFAR-10 Image Classification: Overfitting vs. Regularization

This code demonstrates the impact of **Data Augmentation** and **Dropout** in training Convolutional Neural Networks (CNNs). We compare a baseline model (prone to overfitting) against a regularized model designed for better generalization.

## 1. Load and Prepare the Dataset

We use the CIFAR-10 dataset. We normalize the pixel values to a range of [0, 1] to help the neural network converge faster.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

## 2. Build the Baseline Model

This model consists of standard Convolutional and Pooling layers. Without regularization, this model is likely to memorize the training data (overfit).

In [ ]:
print("--- Building Baseline Model ---")
baseline_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

baseline_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

## 3. Build the Regularized Model

To combat overfitting, we introduce:
1. **Data Augmentation**: Flips, rotations, and zooms.
2. **Dropout**: Randomly ignoring 50% of neurons to prevent over-reliance on specific paths.

In [ ]:
# Define a Data Augmentation pipeline
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal", input_shape=(32, 32, 3)),
  tf.keras.layers.RandomRotation(0.1),
  tf.keras.layers.RandomZoom(0.1),
])

print("--- Building Regularized Model ---")
regularized_model = tf.keras.Sequential([
    data_augmentation,
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

regularized_model.compile(optimizer='adam',
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])

## 4. Train Both Models

We train for 15 epochs to observe the divergence in training and validation accuracy.

In [ ]:
EPOCHS = 15

print("\nTraining Baseline Model...")
baseline_history = baseline_model.fit(train_images, train_labels, epochs=EPOCHS, 
                                      validation_data=(test_images, test_labels), verbose=2)

print("\nTraining Regularized Model...")
regularized_history = regularized_model.fit(train_images, train_labels, epochs=EPOCHS, 
                                            validation_data=(test_images, test_labels), verbose=2)

## 5. Plot and Compare Results

Observe the 'overfitting gap' between training and validation accuracy in both models.

In [ ]:
plt.figure(figsize=(12, 5))

# Plot Baseline Model
plt.subplot(1, 2, 1)
plt.plot(baseline_history.history['accuracy'], label='Train Acc')
plt.plot(baseline_history.history['val_accuracy'], label='Val Acc')
plt.title('Baseline Model')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot Regularized Model
plt.subplot(1, 2, 2)
plt.plot(regularized_history.history['accuracy'], label='Train Acc')
plt.plot(regularized_history.history['val_accuracy'], label='Val Acc')
plt.title('Regularized Model')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()